In [2]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime
 
producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)
 
sklepy = ['Warszawa', 'Krakow', 'Gdansk', 'Wroclaw']
kategorie = ['elektronika', 'odziez', 'zywnosc', 'ksiazki']
 
def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(sklepy),
        'category': random.choice(kategorie),
        'timestamp': datetime.now().isoformat(),
    }
 
for i in range(1000):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}")
    time.sleep(3)
 
producer.flush()
producer.close()

Writing producer.py


In [3]:
#https://sebkaz-teaching.github.io/RTA2026/labs/zaoczne_cw2.html
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pickle
 
np.random.seed(42)
 
# === PARAMETRY — zmień tutaj ===
N_NORMAL = 2000      # liczba normalnych transakcji
N_FRAUD  = 100       # liczba fraudów
# ===============================
 
# Normalne transakcje
normal = pd.DataFrame({
    'amount': np.random.lognormal(5, 1, N_NORMAL).clip(5, 5000),
    'is_electronics': np.random.binomial(1, 0.3, N_NORMAL),
    'tx_per_minute': np.random.poisson(3, N_NORMAL),
    'fraud': 0
})
 
 
# Fraudy
fraud = pd.DataFrame({
    'amount': np.random.uniform(2000, 9000, N_FRAUD),
    'is_electronics': np.random.binomial(1, 0.7, N_FRAUD),
    'tx_per_minute': np.random.poisson(8, N_FRAUD),
    'fraud': 1
})
 
df = pd.concat([normal, fraud], ignore_index=True).sample(frac=1, random_state=42)
print(f"Dataset: {len(df)} wierszy, fraud rate: {df['fraud'].mean():.1%}")

Dataset: 2100 wierszy, fraud rate: 4.8%


In [5]:
features= ['amount', 'is_electronics', 'tx_per_minute']
X= df[features]
y= df['fraud']
# ROZWIĄZANIE

X_train, X_test, y_train, y_test= train_test_split(
X, y, test_size=0.2, stratify=y, random_state=42
)

clf= RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

y_pred= clf.predict(X_test)
print(classification_report(y_test, y_pred))

with open('fraud_model.pkl', 'wb') as f:
    pickle.dump(clf, f)

print("Model zapisany do fraud_model.pkl")

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       400
           1       1.00      1.00      1.00        20

    accuracy                           1.00       420
   macro avg       1.00      1.00      1.00       420
weighted avg       1.00      1.00      1.00       420

Model zapisany do fraud_model.pkl


In [6]:
%%file fraud_api.py
from fastapi import FastAPI
from pydantic import BaseModel
import pickle, numpy as np

app = FastAPI(title="Fraud Detection API")
model = pickle.load(open('fraud_model.pkl', 'rb'))

class Transaction(BaseModel):
    amount: float
    is_electronics: int
    tx_per_minute: int

@app.get("/health")
def health():    
    return {"status": "ok"}
 

Writing fraud_api.py


In [ ]:
# uvicorn fraud_api:app --host 0.0.0.0 --port 8001

In [8]:
import requests

# Test normalna
r = requests.post("http://localhost:8001/score",
    json={"amount": 150, "is_electronics": 0, "tx_per_minute": 3})
print("Normalna:", r.json())

In [9]:
%%file fraud_api.py
from fastapi import FastAPI
from pydantic import BaseModel
import pickle, numpy as np
 
app = FastAPI(title="Fraud Detection API")
 
model = pickle.load(open('fraud_model.pkl', 'rb'))
 
class Transaction(BaseModel):
    amount: float
    is_electronics: int
    tx_per_minute: int
 
@app.post("/score")
def score(tx: Transaction):
    X = np.array([[tx.amount, tx.is_electronics, tx.tx_per_minute]])
    prediction     = model.predict(X)[0]
 
    return {
        **tx,
        "is_fraud":          prediction,
        "model":             "random_forest",
    }
 
@app.get("/health")
def health():
    return {"status": "ok"}

Overwriting fraud_api.py


In [10]:
from sklearn.ensemble import IsolationForest

import pickle
 
features = ['amount', 'is_electronics', 'tx_per_minute']

X_train = normal[features]
 
# contamination: spodziewamy się ~5% anomalii w strumieniu

iso_forest = IsolationForest(

    n_estimators=100,

    contamination=0.05,

    random_state=42

)

iso_forest.fit(X_train)
 
print("Model wytrenowany.")

print(f"Liczba drzew: {iso_forest.n_estimators}")

print(f"Contamination: {iso_forest.contamination}")
 
# Zapisz model

with open('fraud_model_if.pkl', 'wb') as f:

    pickle.dump(iso_forest, f)

print("\nZapisano do fraud_model_if.pkl")
 
%%file fraud_api.py

from fastapi import FastAPI

from pydantic import BaseModel

import pickle

import numpy as np
 
app = FastAPI(title="Fraud Detection API — Isolation Forest")
 
model = pickle.load(open('fraud_model_if.pkl', 'rb'))
 
class Transaction(BaseModel):

    amount: float

    is_electronics: int

    tx_per_minute: int
 
@app.post("/score")

def score(tx: Transaction):

    X = np.array([[tx.amount, tx.is_electronics, tx.tx_per_minute]])

    prediction     = model.predict(X)[0]           # +1 lub -1

    anomaly_score  = model.decision_function(X)[0]  # ujemny = bardziej podejrzany
 
    # Normalizujemy score do przedziału [0, 1] — dla spójności z Ćw. 2

    # decision_function typowo zwraca wartości z zakresu [-0.5, 0.5]

    fraud_probability = float(np.clip(0.5 - anomaly_score, 0.0, 1.0))
 
    return {

        "is_fraud":          bool(prediction == -1),

        "fraud_probability": round(fraud_probability, 4),

        "model":             "isolation_forest",

    }
 
@app.get("/health")

def health():

    return {"status": "ok"}
 
uvicorn fraud_api:app --host 0.0.0.0 --port 8001
 
import requests, time
 
time.sleep(1)  # daj chwilę na restart serwera
 
cases = [

    {"amount": 150,  "is_electronics": 0, "tx_per_minute": 3,  "opis": "normalna"},

    {"amount": 4800, "is_electronics": 1, "tx_per_minute": 12, "opis": "podejrzana"},

    {"amount": 89,   "is_electronics": 0, "tx_per_minute": 2,  "opis": "normalna"},

    {"amount": 3200, "is_electronics": 1, "tx_per_minute": 8,  "opis": "podejrzana"},

]
 
for case in cases:

    payload = {k: v for k, v in case.items() if k != 'opis'}

    r = requests.post("http://localhost:8001/score", json=payload)

    result = r.json()

    print(f"[{case['opis']:10s}] amount={case['amount']:5} "

          f"→ fraud={result['is_fraud']}, prob={result['fraud_probability']:.3f}")
 

SyntaxError: invalid syntax (1325886674.py, line 91)